In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into kernel variables
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to compute min/max timestamp and formatted datetimes
def ts_min_max_info(ts_series):
    if ts_series.empty:
        return {
            'min_ts': None,
            'max_ts': None,
            'min_dt': None,
            'max_dt': None
        }
    min_ts = int(ts_series.min())
    max_ts = int(ts_series.max())
    min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
    max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
    return {
        'min_ts': min_ts,
        'max_ts': max_ts,
        'min_dt': min_dt.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'max_dt': max_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    }

# Summarize a telemetry dataframe (generic for metric/trace/log)
def summarize_df(df, name_col=None):
    row_count = len(df)
    ts_info = ts_min_max_info(df['timestamp'] if 'timestamp' in df.columns else pd.Series([], dtype=int))
    unique_cmdb = pd.Series(df['cmdb_id'].dropna().unique()[:50], name='unique_cmdb_sample')
    unique_cmdb_count = int(df['cmdb_id'].nunique(dropna=True))
    top_names = None
    if name_col and name_col in df.columns:
        top_names = df[name_col].value_counts().head(20).reset_index()
        top_names.columns = [name_col, 'count']
    summary = pd.DataFrame([{
        'rows': row_count,
        'unique_cmdb_count': unique_cmdb_count,
        'min_timestamp': ts_info['min_ts'],
        'max_timestamp': ts_info['max_ts'],
        'min_datetime_UTC+8': ts_info['min_dt'],
        'max_datetime_UTC+8': ts_info['max_dt']
    }])
    return summary, unique_cmdb, top_names

# Create summaries
metric_summary, metric_cmdb_sample, metric_kpi_top20 = summarize_df(metric_df, name_col='kpi_name')
trace_summary, trace_cmdb_sample, trace_top20 = summarize_df(trace_df, name_col='trace_name')
log_summary, log_cmdb_sample, log_top20 = summarize_df(log_df, name_col='log_name')
error_summary, error_cmdb_sample, _ = summarize_df(error_logs_df, name_col=None)

# Overall dataset timestamp range across all files
all_timestamps = pd.concat([
    metric_df['timestamp'] if 'timestamp' in metric_df.columns else pd.Series([], dtype=int),
    trace_df['timestamp'] if 'timestamp' in trace_df.columns else pd.Series([], dtype=int),
    log_df['timestamp'] if 'timestamp' in log_df.columns else pd.Series([], dtype=int),
    error_logs_df['timestamp'] if 'timestamp' in error_logs_df.columns else pd.Series([], dtype=int)
], ignore_index=True)
overall_ts_info = ts_min_max_info(all_timestamps)
overall_range = pd.Series({
    'overall_min_timestamp': overall_ts_info['min_ts'],
    'overall_max_timestamp': overall_ts_info['max_ts'],
    'overall_min_datetime_UTC+8': overall_ts_info['min_dt'],
    'overall_max_datetime_UTC+8': overall_ts_info['max_dt']
})

# Keep outputs compact: show summary rows and top lists (head up to 20) and sampled cmdb ids (up to 50)
metric_summary, metric_kpi_top20, metric_cmdb_sample, trace_summary, trace_top20, trace_cmdb_sample, log_summary, log_top20, log_cmdb_sample, error_summary, error_cmdb_sample, overall_range

```
Out[1]:
```
```python
# Summarize execution results in plain English (compact)
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv: 325,611 rows. Timestamp range 1647847800 - 1647851400 (UTC+8: 2022-03-21 15:30:00 - 2022-03-21 16:30:00). "
    "59 unique cmdb_id values (sample includes: " + ", ".join(metric_cmdb_sample.head(8).astype(str).tolist()) + ", ...). "
    "Top KPIs (top 5): " + "; ".join([f\"{r['kpi_name']} ({int(r['count'])})\" for _, r in metric_kpi_top20.head(5).iterrows()]) + ".\n\n"
    "trace.csv: 36,328 rows. Timestamp range 1647849600 - 1647851340 (UTC+8: 2022-03-21 16:00:00 - 2022-03-21 16:29:00). "
    "40 unique cmdb_id values (sample includes: " + ", ".join(trace_cmdb_sample.head(8).astype(str).tolist()) + ", ...). "
    "Top trace names (top 5): " + "; ".join([f\"{r['trace_name']} ({int(r['count'])})\" for _, r in trace_top20.head(5).iterrows()]) + ".\n\n"
    "log.csv: 1,834 rows. Timestamp range 1647849600 - 1647851400 (UTC+8: 2022-03-21 16:00:00 - 2022-03-21 16:30:00). "
    "31 unique cmdb_id values (sample includes: " + ", ".join(log_cmdb_sample.head(8).astype(str).tolist()) + ", ...). "
    "Top log names: " + "; ".join([f\"{r['log_name']} ({int(r['count'])})\" for _, r in log_top20.iterrows()]) + ".\n\n"
    "error_logs.csv: 0 rows (no timestamps or cmdb_ids present).\n\n"
    "Overall dataset timestamp range across all files: 1647847800 - 1647851400 "
    "(UTC+8: 2022-03-21 15:30:00 - 2022-03-21 16:30:00)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     rows  unique_cmdb_count  min_timestamp  max_timestamp       min_datetime_UTC+8       max_datetime_UTC+8
0  325611                 59     1647847800     1647851400  2022-03-21 15:30:00 CST  2022-03-21 16:30:00 CST,                                              kpi_name  count
0   container.node-6.container_network_transmit_pa...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_transmit_er...   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_transmit_MB...   2196
5   container.node-6.container_network_receive_pac...   2196
6   container.node-6.container_network_receive_err...   2196
7   container.node-6.container_network_receive_MB....   2196
8   container.node-6.container_spec_memory_swap_li...   2074
9   container.node-6.container_fs_io_time_seconds....   2074
10  container.node-6.container_fs_writes_merged./d...   2074
11            container.node-6.container_memory_cache   2074
12  container.node-6.container_memory_failures.hie...   2074
13     container.node-6.container_fs_inodes./dev/vda1   2074
14   container.node-6.container_fs_usage_MB./dev/vda1   2074
15  container.node-6.container_fs_write_seconds./d...   2074
16         container.node-6.container_spec_cpu_period   2074
17                 container.node-6.container_sockets   2074
18          container.node-6.container_spec_cpu_quota   2074
19  container.node-6.container_fs_io_current./dev/...   2074, 0                    adservice
1                  adservice-0
2                  adservice-1
3                  adservice-2
4                   adservice2
                ...           
45     productcatalogservice-2
46    productcatalogservice2-0
47       recommendationservice
48     recommendationservice-0
49     recommendationservice-1
Name: unique_cmdb_sample, Length: 50, dtype: object,     rows  unique_cmdb_count  min_timestamp  max_timestamp       min_datetime_UTC+8       max_datetime_UTC+8
0  36328                 40     1647849600     1647851340  2022-03-21 16:00:00 CST  2022-03-21 16:29:00 CST,                                     trace_name  count
0              trace.from_frontend-1.row_count    630
1             trace.from_frontend-1.error_rate    630
2           trace.from_frontend-1.duration_p95    630
3          trace.from_frontend-1.duration_mean    630
4              trace.from_frontend-0.row_count    602
5             trace.from_frontend-0.error_rate    602
6           trace.from_frontend-0.duration_p95    602
7          trace.from_frontend-0.duration_mean    602
8          trace.from_frontend-2.duration_mean    592
9           trace.from_frontend-2.duration_p95    592
10            trace.from_frontend-2.error_rate    592
11             trace.from_frontend-2.row_count    592
12      trace.from_checkoutservice-0.row_count    539
13     trace.from_checkoutservice-0.error_rate    539
14   trace.from_checkoutservice-0.duration_p95    539
15  trace.from_checkoutservice-0.duration_mean    539
16      trace.from_checkoutservice-2.row_count    539
17     trace.from_checkoutservice-2.error_rate    539
18   trace.from_checkoutservice-2.duration_p95    539
19  trace.from_checkoutservice-2.duration_mean    539, 0                  adservice-0
1                  adservice-1
2                  adservice-2
3                 adservice2-0
4                cartservice-0
                ...           
35    recommendationservice2-0
36           shippingservice-0
37           shippingservice-1
38           shippingservice-2
39          shippingservice2-0
Name: unique_cmdb_sample, Length: 40, dtype: object,    rows  unique_cmdb_count  min_timestamp  max_timestamp       min_datetime_UTC+8       max_datetime_UTC+8
0  1834                 31     1647849600     1647851400  2022-03-21 16:00:00 CST  2022-03-21 16:30:00 CST,           log_name  count
0  log.error_count    917
1    log.row_count    917, 0                 adservice-0
1                 adservice-1
2                 adservice-2
3               cartservice-0
4               cartservice-1
               ...           
26    recommendationservice-2
27          shippingservice-0
28          shippingservice-1
29          shippingservice-2
30               redis-cart-0
Name: unique_cmdb_sample, Length: 31, dtype: object,    rows  unique_cmdb_count min_timestamp max_timestamp min_datetime_UTC+8 max_datetime_UTC+8
0     0                  0          None          None               None               None, Series([], Name: unique_cmdb_sample, dtype: object), overall_min_timestamp                      1647847800
overall_max_timestamp                      1647851400
overall_min_datetime_UTC+8    2022-03-21 15:30:00 CST
overall_max_datetime_UTC+8    2022-03-21 16:30:00 CST
dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df already loaded in kernel
# timezone (UTC+8) variable may exist; ensure available
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95, P05, total_points per (cmdb_id, kpi_name) using full series
group_stats = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        total_points='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter metric rows to the failure window [1647849600, 1647851400]
start_ts = 1647849600
end_ts = 1647851400
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window rows to mark anomalies
window_with_thresh = window_df.merge(group_stats, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

# Mark anomalies: value >= P95 OR value <= P05
window_with_thresh['is_anomaly'] = (window_with_thresh['value'] >= window_with_thresh['P95']) | (window_with_thresh['value'] <= window_with_thresh['P05'])

# Prepare per-group window aggregates
# points_in_window
points_in_window = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(points_in_window='count')
    .reset_index()
)

# anomaly aggregates among anomalies in window
anomaly_rows = window_with_thresh[window_with_thresh['is_anomaly']].copy()

# overall anomaly aggregates (count, earliest timestamp, max/min anomaly value)
if not anomaly_rows.empty:
    anomaly_agg = (
        anomaly_rows
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_anomaly_value=('value', 'max'),
            min_anomaly_value=('value', 'min')
        )
        .reset_index()
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','min_anomaly_value'])

# high-specific and low-specific values for severity calculations
# max value among high anomalies (value >= P95)
high_rows = window_with_thresh[window_with_thresh['value'] >= window_with_thresh['P95']].copy()
if not high_rows.empty:
    high_agg = (
        high_rows
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .agg(max_high_anomaly_value='max')
        .reset_index()
    )
else:
    high_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','max_high_anomaly_value'])

# min value among low anomalies (value <= P05)
low_rows = window_with_thresh[window_with_thresh['value'] <= window_with_thresh['P05']].copy()
if not low_rows.empty:
    low_agg = (
        low_rows
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .agg(min_low_anomaly_value='min')
        .reset_index()
    )
else:
    low_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','min_low_anomaly_value'])

# 4) Merge all aggregates with global group_stats
merged = (
    group_stats
    .merge(points_in_window, on=['cmdb_id','kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
    .merge(high_agg, on=['cmdb_id','kpi_name'], how='left')
    .merge(low_agg, on=['cmdb_id','kpi_name'], how='left')
)

# fillna for counts/points with zeros where appropriate
merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)

# compute severity_high and severity_low per spec
# severity_high = max_high_anomaly_value / P95 (null if P95==0 or no high anomalies)
merged['severity_high'] = np.where(
    (merged.get('max_high_anomaly_value').notna()) & (merged['P95'] != 0),
    merged['max_high_anomaly_value'] / merged['P95'],
    np.nan
)

# severity_low = P05 / min_low_anomaly_value (null if min_low_anomaly_value==0 or no low anomalies)
merged['severity_low'] = np.where(
    (merged.get('min_low_anomaly_value').notna()) & (merged['min_low_anomaly_value'] != 0),
    merged['P05'] / merged['min_low_anomaly_value'],
    np.nan
)

# Ensure earliest_anomaly_timestamp, max/min anomaly values are null when anomaly_count==0
no_anom_mask = merged['anomaly_count'] == 0
for c in ['earliest_anomaly_timestamp','max_anomaly_value','min_anomaly_value','max_high_anomaly_value','min_low_anomaly_value']:
    if c in merged.columns:
        merged.loc[no_anom_mask, c] = np.nan

# 5) Return top 20 groups ordered by anomaly_count desc, then severity_high desc, then severity_low desc
result_cols = [
    'cmdb_id', 'kpi_name', 'total_points', 'P95', 'P05',
    'points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp',
    'max_anomaly_value', 'min_anomaly_value', 'severity_high', 'severity_low'
]

# Some groups might be missing columns for max/min anomaly values if no anomalies existed; ensure columns exist
for col in ['max_anomaly_value','min_anomaly_value']:
    if col not in merged.columns:
        merged[col] = np.nan

# sort and select top 20
result_df = (
    merged
    .sort_values(by=['anomaly_count', 'severity_high', 'severity_low'], ascending=[False, False, False])
    .loc[:, result_cols]
    .head(20)
)

# Compact formatting: round numeric thresholds and severities for readability
for col in ['P95','P05','max_anomaly_value','min_anomaly_value','severity_high','severity_low']:
    if col in result_df.columns:
        result_df[col] = result_df[col].round(6)

# Display compact result (max 20 rows)
result_df

```
Out[2]:
```
```python
# Summarize the previous execution results in plain English (compact)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse `merged` and `result_df` from prior execution
# compute total groups with any anomalies (using merged computed earlier)
total_groups_with_anomalies = int((merged['anomaly_count'] > 0).sum())

# prepare top 5 groups from the result table for a concise summary
top_n = result_df.head(5).copy()
def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

rows = []
for _, r in top_n.iterrows():
    rows.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | "
        f"P95={r['P95']} | P05={r['P05']} | earliest={fmt_ts(r['earliest_anomaly_timestamp'])} | "
        f"severity_high={r['severity_high'] if not pd.isna(r['severity_high']) else 'N/A'} | "
        f"severity_low={r['severity_low'] if not pd.isna(r['severity_low']) else 'N/A'}"
    )

summary_str = (
    f"Metric anomaly summary for window 1647849600–1647851400 (UTC+8):\n"
    f"- Total (cmdb_id, kpi_name) groups with >=1 anomaly in window: {total_groups_with_anomalies}\n"
    f"- Top {len(rows)} groups by anomaly_count (compact):\n  - " + "\n  - ".join(rows) + "\n\n"
    "Notes: P95/P05 were computed globally per group using the full series before window filtering. "
    "Severities: severity_high = max_anomaly_value / P95 (N/A if P95==0 or no high anomalies); "
    "severity_low = P05 / min_anomaly_value (N/A if min==0 or no low anomalies)."
)

summary_str
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points           P95         P05  points_in_window  anomaly_count  earliest_anomaly_timestamp  max_anomaly_value  min_anomaly_value  severity_high  severity_low
2884         frontend-1  container.node-5.container_cpu_cfs_throttled_s...            61  0.000000e+00         0.0                31             31                1.647850e+09       3.373000e-03                0.0   3.709033e+09           NaN
4048  paymentservice2-0      container.node-6.container_cpu_system_seconds            61  5.000000e-03         0.0                31             31                1.647850e+09       2.650000e-01                0.0   5.300000e+01           NaN
3998   paymentservice-2            container.node-6.container_memory_cache            61  1.429504e+06   1429504.0                31             31                1.647850e+09       2.427290e+07          1429504.0   1.697994e+01           1.0
3934   paymentservice-1      container.node-6.container_memory_mapped_file            61  1.351680e+05    135168.0                31             31                1.647850e+09       1.486848e+06           135168.0   1.100000e+01           1.0
1352     cartservice2-0  container.node-5.container_cpu_cfs_throttled_s...            61  1.675400e-02         0.0                31             31                1.647850e+09       1.224100e-01                0.0   7.306290e+00           NaN
2605     emailservice-2  mesh.source.emailservice.jaeger-collector.isti...            61  5.250000e-04         0.0                31             31                1.647850e+09       2.050000e-03                0.0   3.904762e+00           NaN
4031   paymentservice-2                 container.node-6.container_threads            61  1.900000e+01        19.0                31             31                1.647850e+09       4.800000e+01               19.0   2.526316e+00           1.0
3891   paymentservice-0                 container.node-6.container_threads            61  2.200000e+01        22.0                31             31                1.647850e+09       5.100000e+01               22.0   2.318182e+00           1.0
3961   paymentservice-1                 container.node-6.container_threads            61  2.300000e+01        23.0                31             31                1.647850e+09       5.200000e+01               23.0   2.260870e+00           1.0
4105  paymentservice2-0                 container.node-6.container_threads            61  2.500000e+01        25.0                31             31                1.647850e+09       5.300000e+01               25.0   2.120000e+00           1.0
2124  currencyservice-1  container.node-6.container_cpu_cfs_throttled_s...            61  9.408000e-03         0.0                31             31                1.647850e+09       1.897200e-02                0.0   2.016577e+00           NaN
1351     cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0                31             31                1.647850e+09       1.000000e+00                0.0   2.000000e+00           NaN
2043  currencyservice-0  container.node-6.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0                31             31                1.647850e+09       1.000000e+00                0.0   2.000000e+00           NaN
2203  currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0                31             31                1.647850e+09       1.000000e+00                0.0   2.000000e+00           NaN
3866   paymentservice-0              container.node-6.container_memory_rss            61  3.370598e+07  33591296.0                31             31                1.647850e+09       5.110170e+07         33591296.0   1.516102e+00           1.0
4006   paymentservice-2              container.node-6.container_memory_rss            61  3.399680e+07  33996800.0                31             31                1.647850e+09       5.147443e+07         33996800.0   1.514096e+00           1.0
3665             node-4                                    system.io.await            61  5.000000e-01         0.0                31             31                1.647850e+09       7.500000e-01                0.0   1.500000e+00           NaN
3671             node-4                                  system.io.w_await            61  5.000000e-01         0.0                31             31                1.647850e+09       7.500000e-01                0.0   1.500000e+00           NaN
3837   paymentservice-0        container.node-6.container_file_descriptors            61  2.100000e+01        21.0                31             31                1.647850e+09       3.100000e+01               21.0   1.476190e+00           1.0
3907   paymentservice-1        container.node-6.container_file_descriptors            61  2.200000e+01        22.0                31             31                1.647850e+09       3.200000e+01               22.0   1.454545e+00           1.0```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables: metric_df, merged, window_df (if not, recreate window_df)
start_ts = 1647849600
end_ts = 1647851400
tz = pytz.timezone('Asia/Shanghai')

# Ensure window_df exists
try:
    window_df
except NameError:
    window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Select top 12 groups from merged ordered by anomaly_count desc, severity_high desc, severity_low desc
top_groups = (
    merged
    .sort_values(by=['anomaly_count', 'severity_high', 'severity_low'], ascending=[False, False, False])
    .head(12)
    [['cmdb_id','kpi_name','total_points','P95','P05','points_in_window','anomaly_count']]
    .copy()
)

# Helper to analyze one group
def analyze_group(cmdb_id, kpi_name, P95, P05):
    gdf = window_df[(window_df['cmdb_id'] == cmdb_id) & (window_df['kpi_name'] == kpi_name)].sort_values('timestamp').reset_index(drop=True)
    points_in_window = len(gdf)
    if points_in_window == 0:
        return {
            'points_in_window': 0,
            'anomaly_count': 0,
            'num_segments': 0,
            'longest_segment_length': 0,
            'longest_segment_start_timestamp': np.nan,
            'longest_segment_end_timestamp': np.nan,
            'longest_segment_max_value': np.nan,
            'longest_segment_min_value': np.nan,
            'breach_ratio_high': np.nan,
            'breach_ratio_low': np.nan
        }
    # mark anomalies using global P95/P05
    gdf['is_anom'] = (gdf['value'] >= P95) | (gdf['value'] <= P05)
    anomaly_count = int(gdf['is_anom'].sum())
    if anomaly_count == 0:
        return {
            'points_in_window': points_in_window,
            'anomaly_count': 0,
            'num_segments': 0,
            'longest_segment_length': 0,
            'longest_segment_start_timestamp': np.nan,
            'longest_segment_end_timestamp': np.nan,
            'longest_segment_max_value': np.nan,
            'longest_segment_min_value': np.nan,
            'breach_ratio_high': np.nan,
            'breach_ratio_low': np.nan
        }
    # find segments of consecutive anomalous rows in the window-ordered sequence
    anom_idx = np.flatnonzero(gdf['is_anom'].values)
    # split into segments where indices are not consecutive
    seg_starts = [anom_idx[0]]
    seg_ends = []
    for prev, curr in zip(anom_idx[:-1], anom_idx[1:]):
        if curr != prev + 1:
            seg_ends.append(prev)
            seg_starts.append(curr)
    seg_ends.append(anom_idx[-1])
    num_segments = len(seg_starts)
    # compute lengths and stats per segment
    seg_lengths = []
    seg_stats = []
    for s, e in zip(seg_starts, seg_ends):
        seg_slice = gdf.iloc[s:e+1]
        length = len(seg_slice)
        seg_lengths.append(length)
        seg_stats.append({
            'start_ts': int(seg_slice['timestamp'].iloc[0]),
            'end_ts': int(seg_slice['timestamp'].iloc[-1]),
            'max_value': float(seg_slice['value'].max()),
            'min_value': float(seg_slice['value'].min()),
            'has_high': bool((seg_slice['value'] >= P95).any()),
            'has_low': bool((seg_slice['value'] <= P05).any())
        })
    # identify longest segment (if tie, pick the earliest by start timestamp)
    max_len = max(seg_lengths)
    candidates = [i for i,l in enumerate(seg_lengths) if l == max_len]
    chosen_idx = candidates[0]
    chosen = seg_stats[chosen_idx]
    # compute breach ratios for longest segment
    breach_ratio_high = np.nan
    breach_ratio_low = np.nan
    if chosen['has_high'] and P95 != 0:
        breach_ratio_high = chosen['max_value'] / P95
    if chosen['has_low'] and chosen['min_value'] != 0:
        # P05 might be zero; per spec only compute if P05>0
        if P05 != 0:
            breach_ratio_low = P05 / chosen['min_value']
        else:
            breach_ratio_low = np.nan
    return {
        'points_in_window': points_in_window,
        'anomaly_count': anomaly_count,
        'num_segments': num_segments,
        'longest_segment_length': int(max_len),
        'longest_segment_start_timestamp': chosen['start_ts'],
        'longest_segment_end_timestamp': chosen['end_ts'],
        'longest_segment_max_value': chosen['max_value'],
        'longest_segment_min_value': chosen['min_value'],
        'breach_ratio_high': breach_ratio_high,
        'breach_ratio_low': breach_ratio_low
    }

# Apply analysis to top groups
rows = []
for _, row in top_groups.iterrows():
    cmdb_id = row['cmdb_id']
    kpi_name = row['kpi_name']
    P95 = row['P95']
    P05 = row['P05']
    stats = analyze_group(cmdb_id, kpi_name, P95, P05)
    out = {
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'total_points': int(row['total_points']),
        'points_in_window': int(stats['points_in_window']),
        'anomaly_count': int(stats['anomaly_count']),
        'num_segments': int(stats['num_segments']),
        'longest_segment_length': int(stats['longest_segment_length']),
        'longest_segment_start_timestamp': stats['longest_segment_start_timestamp'],
        'longest_segment_end_timestamp': stats['longest_segment_end_timestamp'],
        'longest_segment_max_value': stats['longest_segment_max_value'],
        'longest_segment_min_value': stats['longest_segment_min_value'],
        'breach_ratio_high': stats['breach_ratio_high'],
        'breach_ratio_low': stats['breach_ratio_low']
    }
    rows.append(out)

result_top12 = pd.DataFrame(rows)

# Sort as requested: anomaly_count desc then breach_ratio_high desc
result_top12 = result_top12.sort_values(by=['anomaly_count','breach_ratio_high'], ascending=[False, False]).reset_index(drop=True)

# Round numeric columns for compactness
for c in ['longest_segment_max_value','longest_segment_min_value','breach_ratio_high','breach_ratio_low']:
    if c in result_top12.columns:
        result_top12[c] = result_top12[c].round(6)

# Display compact table (max 12 rows)
result_top12

```
Out[3]:
```
```python
# Compact plain-English summary of the previous analysis (uses result_top12 from kernel)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use result_top12 computed earlier
df = result_top12.copy()

# Helper to format timestamp to UTC+8 readable
def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Basic high-level observations (many top groups share same window behavior)
common_total = df['total_points'].mode().iloc[0]
common_points_in_window = df['points_in_window'].mode().iloc[0]
common_anomaly_count = df['anomaly_count'].mode().iloc[0]
common_num_segments = df['num_segments'].mode().iloc[0]
common_longest_len = df['longest_segment_length'].mode().iloc[0]
window_start = fmt_ts(df['longest_segment_start_timestamp'].iloc[0])
window_end = fmt_ts(df['longest_segment_end_timestamp'].iloc[0])

# Top breach_ratio_high performers (non-null), show up to 5
br_high = df[df['breach_ratio_high'].notna()].sort_values('breach_ratio_high', ascending=False).head(5)
top_high_lines = []
for _, r in br_high.iterrows():
    top_high_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | breach_ratio_high={r['breach_ratio_high']} | longest_max={r['longest_segment_max_value']}")

# Compose summary string
summary = (
    "Summary of segment analysis for top 12 anomalous (cmdb_id, kpi_name) groups:\n\n"
    f"- Common facts for these top 12 groups:\n"
    f"  • total_points (global): {common_total}\n"
    f"  • points_in_window: {common_points_in_window} (window {window_start} to {window_end})\n"
    f"  • anomaly_count in window: {common_anomaly_count}\n"
    f"  • num_segments: {common_num_segments} (i.e., a single continuous anomalous segment)\n"
    f"  • longest_segment_length: {common_longest_len} (the segment spans the entire selected window)\n\n"
    "- Interpretation: For each of these top groups, anomalies occurred continuously throughout the failure window (every sampled minute in-window was anomalous), producing one long segment covering the whole window.\n\n"
    "- Top groups by breach_ratio_high (up to 5):\n  - " + ("\n  - ".join(top_high_lines) if top_high_lines else "None with high-breach ratio available") + "\n\n"
    "- Notes:\n"
    "  • breach_ratio_high = longest_segment_max_value / P95 (shown only when P95>0 and high anomalies exist).\n"
    "  • breach_ratio_low = P05 / longest_segment_min_value (shown only when P05>0 and low anomalies exist).\n"
    "  • Many groups have extremely large breach_ratio_high values or NaN for breach_ratio_low when P05==0 or no low anomalies.\n\n"
    "Conclusion: The top 12 groups all exhibited persistent anomalies covering the entire window; triage should focus on high breach_ratio_high cases first (listed above) and on common resource types (container cpu/memory/thread/file-descriptor metrics) for root cause investigation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  points_in_window  anomaly_count  num_segments  longest_segment_length  longest_segment_start_timestamp  longest_segment_end_timestamp  longest_segment_max_value  longest_segment_min_value  breach_ratio_high  breach_ratio_low
0          frontend-1  container.node-5.container_cpu_cfs_throttled_s...            61                31             31             1                      31                       1647849600                     1647851400               3.373000e-03                        0.0       3.709033e+09               NaN
1   paymentservice2-0      container.node-6.container_cpu_system_seconds            61                31             31             1                      31                       1647849600                     1647851400               2.650000e-01                        0.0       5.300000e+01               NaN
2    paymentservice-2            container.node-6.container_memory_cache            61                31             31             1                      31                       1647849600                     1647851400               2.427290e+07                  1429504.0       1.697994e+01               1.0
3    paymentservice-1      container.node-6.container_memory_mapped_file            61                31             31             1                      31                       1647849600                     1647851400               1.486848e+06                   135168.0       1.100000e+01               1.0
4      cartservice2-0  container.node-5.container_cpu_cfs_throttled_s...            61                31             31             1                      31                       1647849600                     1647851400               1.224100e-01                        0.0       7.306290e+00               NaN
5      emailservice-2  mesh.source.emailservice.jaeger-collector.isti...            61                31             31             1                      31                       1647849600                     1647851400               2.050000e-03                        0.0       3.904762e+00               NaN
6    paymentservice-2                 container.node-6.container_threads            61                31             31             1                      31                       1647849600                     1647851400               4.800000e+01                       19.0       2.526316e+00               1.0
7    paymentservice-0                 container.node-6.container_threads            61                31             31             1                      31                       1647849600                     1647851400               5.100000e+01                       22.0       2.318182e+00               1.0
8    paymentservice-1                 container.node-6.container_threads            61                31             31             1                      31                       1647849600                     1647851400               5.200000e+01                       23.0       2.260870e+00               1.0
9   paymentservice2-0                 container.node-6.container_threads            61                31             31             1                      31                       1647849600                     1647851400               5.300000e+01                       25.0       2.120000e+00               1.0
10  currencyservice-1  container.node-6.container_cpu_cfs_throttled_s...            61                31             31             1                      31                       1647849600                     1647851400               1.897200e-02                        0.0       2.016577e+00               NaN
11     cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...            61                31             31             1                      31                       1647849600                     1647851400               1.000000e+00                        0.0       2.000000e+00               NaN```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df if present
tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95, P05, total_points per (cmdb_id, trace_name) using full series (do NOT filter first)
trace_group_stats = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        total_points='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter trace rows to the given window [1647849600, 1647851400]
start_ts = 1647849600
end_ts = 1647851400
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge global thresholds into window rows and mark anomalies
tw_with_thresh = trace_window.merge(trace_group_stats, on=['cmdb_id','trace_name'], how='left', validate='m:1')
tw_with_thresh['is_anomaly'] = (tw_with_thresh['value'] >= tw_with_thresh['P95']) | (tw_with_thresh['value'] <= tw_with_thresh['P05'])

# points_in_window per group
points_in_window = (
    trace_window
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(points_in_window='count')
    .reset_index()
)

# anomaly aggregates among anomaly rows in window
anomaly_rows = tw_with_thresh[tw_with_thresh['is_anomaly']].copy()
if not anomaly_rows.empty:
    anomaly_agg = (
        anomaly_rows
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_value=('value','max'),
            min_value=('value','min')
        )
        .reset_index()
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','min_value'])

# detect presence of high/low anomalies per group within window
high_rows = tw_with_thresh[tw_with_thresh['value'] >= tw_with_thresh['P95']].copy()
if not high_rows.empty:
    high_exist = high_rows.groupby(['cmdb_id','trace_name'], dropna=False)['value'].agg(has_high='count').reset_index()[['cmdb_id','trace_name','has_high']]
else:
    high_exist = pd.DataFrame(columns=['cmdb_id','trace_name','has_high'])

low_rows = tw_with_thresh[tw_with_thresh['value'] <= tw_with_thresh['P05']].copy()
if not low_rows.empty:
    low_exist = low_rows.groupby(['cmdb_id','trace_name'], dropna=False)['value'].agg(has_low='count').reset_index()[['cmdb_id','trace_name','has_low']]
else:
    low_exist = pd.DataFrame(columns=['cmdb_id','trace_name','has_low'])

# 4) Merge aggregates back into group stats
merged_trace = (
    trace_group_stats
    .merge(points_in_window, on=['cmdb_id','trace_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','trace_name'], how='left')
    .merge(high_exist, on=['cmdb_id','trace_name'], how='left')
    .merge(low_exist, on=['cmdb_id','trace_name'], how='left')
)

# Fill missing numeric counts with zeros
merged_trace['points_in_window'] = merged_trace['points_in_window'].fillna(0).astype(int)
merged_trace['anomaly_count'] = merged_trace['anomaly_count'].fillna(0).astype(int)
merged_trace['has_high'] = merged_trace.get('has_high').fillna(0).astype(int) if 'has_high' in merged_trace.columns else 0
merged_trace['has_low'] = merged_trace.get('has_low').fillna(0).astype(int) if 'has_low' in merged_trace.columns else 0

# Ensure max_value/min_value and earliest timestamp are NaN when no anomalies
no_anom_mask = merged_trace['anomaly_count'] == 0
for c in ['earliest_anomaly_timestamp','max_value','min_value']:
    if c in merged_trace.columns:
        merged_trace.loc[no_anom_mask, c] = np.nan

# 4) Compute severity_high and severity_low per spec
# severity_high = max_value / P95 (null if P95==0 or no high anomalies)
merged_trace['severity_high'] = np.where(
    (merged_trace.get('max_value').notna()) & (merged_trace['P95'] != 0) & (merged_trace.get('has_high',0) > 0),
    merged_trace['max_value'] / merged_trace['P95'],
    np.nan
)

# severity_low = P05 / min_value (null if min_value==0 or no low anomalies)
merged_trace['severity_low'] = np.where(
    (merged_trace.get('min_value').notna()) & (merged_trace['min_value'] != 0) & (merged_trace.get('has_low',0) > 0),
    merged_trace['P05'] / merged_trace['min_value'],
    np.nan
)

# 5) Select and sort top 20 groups by anomaly_count desc, then severity_high desc, then severity_low desc
result_cols = [
    'cmdb_id','trace_name','total_points','P95','P05',
    'points_in_window','anomaly_count','earliest_anomaly_timestamp',
    'max_value','min_value','severity_high','severity_low'
]

# Ensure columns exist
for c in result_cols:
    if c not in merged_trace.columns:
        merged_trace[c] = np.nan

result_trace = (
    merged_trace
    .loc[:, result_cols]
    .sort_values(by=['anomaly_count','severity_high','severity_low'], ascending=[False, False, False])
    .head(20)
    .reset_index(drop=True)
)

# Round numeric columns for compact display
for col in ['P95','P05','max_value','min_value','severity_high','severity_low']:
    if col in result_trace.columns:
        result_trace[col] = result_trace[col].round(6)

# Display the compact result (max 20 rows)
result_trace

```
Out[4]:
```
```python
# Plain-English compact summary of the trace analysis (uses result_trace from kernel)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use result_trace computed earlier
df = result_trace.copy()

def fmt_ts(x):
    if pd.isna(x):
        return "N/A"
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# High-level observations
total_groups = len(df)
all_points_in_window = df['points_in_window'].unique().tolist()
all_anomaly_counts = df['anomaly_count'].unique().tolist()

# Top groups by severity_high (non-null)
top_by_high = df[df['severity_high'].notna()].sort_values('severity_high', ascending=False).head(5)
top_high_lines = []
for _, r in top_by_high.iterrows():
    top_high_lines.append(
        f"{r['cmdb_id']} | {r['trace_name']} | P95={r['P95']} | max={r['max_value']} | severity_high={r['severity_high']}"
    )

summary = (
    f"Trace analysis summary (top {total_groups} groups inspected):\n\n"
    f"- Window behavior: All top groups show points_in_window={all_points_in_window} and anomaly_count={all_anomaly_counts}.\n"
    "  This indicates every sampled point in the failure window was flagged anomalous for these groups.\n\n"
    "- Common pattern:\n"
    "  • Many trace duration metrics (e.g., trace.*.duration_mean/p95) have small P95 but observed max values multiple× larger (severity_high > 1).\n"
    "  • Several trace row_count metrics have integer P95/P05 (e.g., P95=2, P05=1) and observed max=3, giving moderate severity_high ~1.5 and severity_low = 1.0.\n"
    "  • Many groups have P05 == 0 or min_value == 0, so severity_low is not available (NaN) for those.\n\n"
    "- Top groups by severity_high (up to 5):\n  - " + ("\n  - ".join(top_high_lines) if top_high_lines else "None with non-null severity_high") + "\n\n"
    "- Interpretation & recommendation:\n"
    "  • The examined trace groups were persistently anomalous across the entire window; investigate services along the checkout→cart/purchase traces first.\n"
    "  • Prioritize groups with largest severity_high (listed above) because their peak values exceed typical P95 by the largest multiples.\n"
    "  • For groups where severity_low is NaN due to zero P05/min, consider separate handling (zero baselines) and inspect raw traces/logs.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  total_points       P95  P05  points_in_window  anomaly_count  earliest_anomaly_timestamp  max_value  min_value  severity_high  severity_low
0             cartservice-0   trace.from_checkoutservice-1.duration_p95            30  0.000495  0.0                30             30                1.647850e+09   0.002550        0.0       5.151515           NaN
1         checkoutservice-1         trace.to_cartservice-0.duration_p95            30  0.000495  0.0                30             30                1.647850e+09   0.002550        0.0       5.151515           NaN
2             cartservice-0  trace.from_checkoutservice-1.duration_mean            30  0.000183  0.0                30             30                1.647850e+09   0.000750        0.0       4.095004           NaN
3         checkoutservice-1        trace.to_cartservice-0.duration_mean            30  0.000183  0.0                30             30                1.647850e+09   0.000750        0.0       4.095004           NaN
4             cartservice-0  trace.from_checkoutservice-2.duration_mean            30  0.000183  0.0                30             30                1.647850e+09   0.000667        0.0       3.641824           NaN
5         checkoutservice-2        trace.to_cartservice-0.duration_mean            30  0.000183  0.0                30             30                1.647850e+09   0.000667        0.0       3.641824           NaN
6             cartservice-0   trace.from_checkoutservice-2.duration_p95            30  0.000495  0.0                30             30                1.647850e+09   0.001800        0.0       3.636364           NaN
7         checkoutservice-2         trace.to_cartservice-0.duration_p95            30  0.000495  0.0                30             30                1.647850e+09   0.001800        0.0       3.636364           NaN
8         checkoutservice-0         trace.to_paymentservice-0.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
9         checkoutservice-0         trace.to_paymentservice-1.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
10        checkoutservice-1         trace.to_paymentservice-0.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
11        checkoutservice-2         trace.to_paymentservice-1.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
12        checkoutservice-2         trace.to_paymentservice-2.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
13         paymentservice-0      trace.from_checkoutservice-0.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
14         paymentservice-0      trace.from_checkoutservice-1.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
15         paymentservice-1      trace.from_checkoutservice-0.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
16         paymentservice-1      trace.from_checkoutservice-2.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
17         paymentservice-2      trace.from_checkoutservice-2.row_count            30  2.000000  1.0                30             30                1.647850e+09   3.000000        1.0       1.500000           1.0
18        checkoutservice-2  trace.to_productcatalogservice-0.row_count            30  3.000000  2.0                30             30                1.647850e+09   4.000000        1.0       1.333333           2.0
19  productcatalogservice-0      trace.from_checkoutservice-2.row_count            30  3.000000  2.0                30             30                1.647850e+09   4.000000        1.0       1.333333           2.0```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse log_df from kernel
start_ts = 1647849600
end_ts = 1647851400
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter log rows to the window
window_logs = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 2) Aggregations by cmdb_id
total_rows = window_logs.groupby('cmdb_id').size().rename('total_rows_in_window')

sum_error = (
    window_logs[window_logs['log_name'] == 'log.error_count']
    .groupby('cmdb_id')['value']
    .sum()
    .rename('sum_log_error_count')
)

sum_row = (
    window_logs[window_logs['log_name'] == 'log.row_count']
    .groupby('cmdb_id')['value']
    .sum()
    .rename('sum_log_row_count')
)

earliest_err_ts = (
    window_logs[(window_logs['log_name'] == 'log.error_count') & (window_logs['value'] > 0)]
    .groupby('cmdb_id')['timestamp']
    .min()
    .rename('earliest_error_timestamp')
)

max_err = (
    window_logs[window_logs['log_name'] == 'log.error_count']
    .groupby('cmdb_id')['value']
    .max()
    .rename('max_log_error_count')
)

# 3) Combine aggregates into one DataFrame
agg_df = pd.concat([total_rows, sum_error, sum_row, earliest_err_ts, max_err], axis=1).reset_index()

# Fill NaN sums with 0 (no rows of that log_name)
agg_df['sum_log_error_count'] = agg_df['sum_log_error_count'].fillna(0)
agg_df['sum_log_row_count'] = agg_df['sum_log_row_count'].fillna(0)

# Keep earliest_error_timestamp and max_log_error_count as NaN when absent (per spec)
# Ensure numeric types for compactness
agg_df['total_rows_in_window'] = agg_df['total_rows_in_window'].astype(int)
agg_df['sum_log_error_count'] = agg_df['sum_log_error_count'].astype(float)
agg_df['sum_log_row_count'] = agg_df['sum_log_row_count'].astype(float)

# 4) Order and limit to top 20
result_logs = (
    agg_df
    .sort_values(by=['sum_log_error_count', 'sum_log_row_count'], ascending=[False, False])
    .loc[:, ['cmdb_id','total_rows_in_window','sum_log_error_count','sum_log_row_count','earliest_error_timestamp','max_log_error_count']]
    .head(20)
    .reset_index(drop=True)
)

# 5) Summary line: number of distinct cmdb_id that had sum_log_error_count > 0 within the window
erroring_cmdb_count = int((agg_df['sum_log_error_count'] > 0).sum())

# Display compact results: result_logs and the single summary integer
result_logs, erroring_cmdb_count

```
Out[5]:
```
```python
# Compact plain-English summary of the log aggregation results (uses result_logs and erroring_cmdb_count from kernel)
summary = (
    "Log aggregation (window 1647849600–1647851400 UTC+8):\n\n"
    "- Top 20 cmdb_id by sum_log_error_count (then sum_log_row_count):\n"
    "  • frontend-1: total_rows_in_window=62, sum_log_row_count=99977, sum_log_error_count=0\n"
    "  • productcatalogservice-1: total_rows_in_window=60, sum_log_row_count=34080, sum_log_error_count=0\n"
    "  • productcatalogservice-0: total_rows_in_window=60, sum_log_row_count=33937, sum_log_error_count=0\n"
    "  • productcatalogservice-2: total_rows_in_window=60, sum_log_row_count=33927, sum_log_error_count=0\n"
    "  • cartservice-0/1/2: each total_rows_in_window≈62, sum_log_row_count≈30226/30204/30203, sum_log_error_count=0\n"
    "  (and others in the top 20 show the same pattern: nonzero row counts but sum_log_error_count=0)\n\n"
    f"- Summary: number of distinct cmdb_id with sum_log_error_count > 0 in the window = {erroring_cmdb_count}.\n"
    "- Interpretation: No log.error_count > 0 was observed for any component in this window; logs show activity (row counts) but no recorded error counts."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  total_rows_in_window  sum_log_error_count  sum_log_row_count  earliest_error_timestamp  max_log_error_count
0                frontend-1                    62                  0.0            99977.0                       NaN                  0.0
1   productcatalogservice-1                    60                  0.0            34080.0                       NaN                  0.0
2   productcatalogservice-0                    60                  0.0            33937.0                       NaN                  0.0
3   productcatalogservice-2                    60                  0.0            33927.0                       NaN                  0.0
4             cartservice-0                    62                  0.0            30226.0                       NaN                  0.0
5             cartservice-1                    62                  0.0            30204.0                       NaN                  0.0
6             cartservice-2                    62                  0.0            30203.0                       NaN                  0.0
7         currencyservice-2                    62                  0.0            26277.0                       NaN                  0.0
8         currencyservice-0                    62                  0.0            26221.0                       NaN                  0.0
9         currencyservice-1                    62                  0.0            26214.0                       NaN                  0.0
10               frontend-0                    60                  0.0            23812.0                       NaN                  0.0
11               frontend-2                    60                  0.0            19982.0                       NaN                  0.0
12  recommendationservice-2                    62                  0.0             7165.0                       NaN                  0.0
13  recommendationservice-1                    62                  0.0             7161.0                       NaN                  0.0
14  recommendationservice-0                    62                  0.0             7156.0                       NaN                  0.0
15              adservice-0                    60                  0.0             4150.0                       NaN                  0.0
16              adservice-1                    60                  0.0             3786.0                       NaN                  0.0
17              adservice-2                    60                  0.0             3786.0                       NaN                  0.0
18        shippingservice-1                    60                  0.0             3288.0                       NaN                  0.0
19        shippingservice-2                    60                  0.0             3278.0                       NaN                  0.0, 0)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Candidates
candidates = ["paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0","node-6","frontend-1","cartservice2-0"]

# Window
start_ts = 1647849600
end_ts = 1647851400

# Recompute global thresholds for metric (per rules: compute BEFORE filtering)
metric_group_stats = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(
        total_points='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Filter metric rows to window
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Merge thresholds into window metric rows
metric_win_with_thresh = metric_window.merge(metric_group_stats, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Mark anomaly rows and compute breach info for metric rows
def compute_metric_breach_info(df):
    # determine breach type: prefer 'high' when both conditions true
    df['is_high'] = df['value'] >= df['P95']
    df['is_low'] = df['value'] <= df['P05']
    df['is_anom'] = df['is_high'] | df['is_low']
    # breach_type
    df['metric_breach_type'] = np.where(df['is_high'], 'high', np.where(df['is_low'], 'low', None))
    # compute breach_ratio depending on type
    def br(row):
        if not row['is_anom']:
            return np.nan
        if row['is_high']:
            if row['P95'] != 0 and not pd.isna(row['P95']):
                return row['value'] / row['P95']
            else:
                return np.nan
        else:  # low
            if row['P05'] != 0 and not pd.isna(row['P05']):
                # per spec: P05 / value
                return row['P05'] / row['value'] if row['value'] != 0 else np.nan
            else:
                return np.nan
    df['metric_breach_ratio'] = df.apply(br, axis=1)
    return df

metric_win_with_thresh = compute_metric_breach_info(metric_win_with_thresh)

# For each candidate, find earliest anomalous metric row (if any)
metric_anoms = metric_win_with_thresh[metric_win_with_thresh['is_anom']].copy()
if metric_anoms.empty:
    metric_earliest = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_timestamp','metric_kpi_name','metric_kpi_value','metric_breach_type','metric_breach_ratio'])
else:
    # sort to pick earliest timestamp, tie-breaker: larger breach_ratio
    metric_anoms_sorted = metric_anoms.sort_values(by=['timestamp','metric_breach_ratio'], ascending=[True, False])
    metric_earliest = (
        metric_anoms_sorted
        .groupby('cmdb_id', sort=False, as_index=False)
        .nth(0)[['cmdb_id','timestamp','kpi_name','value','metric_breach_type','metric_breach_ratio']]
        .rename(columns={
            'timestamp':'earliest_metric_anomaly_timestamp',
            'kpi_name':'metric_kpi_name',
            'value':'metric_kpi_value'
        })
    )

# Recompute global thresholds for traces (before filtering)
trace_group_stats = (
    trace_df
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        total_points='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Filter trace rows to window
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# Merge thresholds into window trace rows
trace_win_with_thresh = trace_window.merge(trace_group_stats, on=['cmdb_id','trace_name'], how='left', validate='m:1')

# Mark anomalies and compute breach info for trace rows
def compute_trace_breach_info(df):
    df['is_high'] = df['value'] >= df['P95']
    df['is_low'] = df['value'] <= df['P05']
    df['is_anom'] = df['is_high'] | df['is_low']
    df['trace_breach_type'] = np.where(df['is_high'], 'high', np.where(df['is_low'], 'low', None))
    def br(row):
        if not row['is_anom']:
            return np.nan
        if row['is_high']:
            if row['P95'] != 0 and not pd.isna(row['P95']):
                return row['value'] / row['P95']
            else:
                return np.nan
        else:
            if row['P05'] != 0 and not pd.isna(row['P05']):
                return row['P05'] / row['value'] if row['value'] != 0 else np.nan
            else:
                return np.nan
    df['trace_breach_ratio'] = df.apply(br, axis=1)
    return df

trace_win_with_thresh = compute_trace_breach_info(trace_win_with_thresh)

# For each candidate, find earliest anomalous trace row (if any)
trace_anoms = trace_win_with_thresh[trace_win_with_thresh['is_anom']].copy()
if trace_anoms.empty:
    trace_earliest = pd.DataFrame(columns=['cmdb_id','earliest_trace_anomaly_timestamp','trace_name','trace_value','trace_breach_type','trace_breach_ratio'])
else:
    trace_anoms_sorted = trace_anoms.sort_values(by=['timestamp','trace_breach_ratio'], ascending=[True, False])
    trace_earliest = (
        trace_anoms_sorted
        .groupby('cmdb_id', sort=False, as_index=False)
        .nth(0)[['cmdb_id','timestamp','trace_name','value','trace_breach_type','trace_breach_ratio']]
        .rename(columns={
            'timestamp':'earliest_trace_anomaly_timestamp',
            'value':'trace_value'
        })
    )

# Prepare final summary table for provided candidates
c_df = pd.DataFrame({'cmdb_id': candidates})

# Merge metric earliest info
c_df = c_df.merge(metric_earliest, on='cmdb_id', how='left')

# Merge trace earliest info
c_df = c_df.merge(trace_earliest, on='cmdb_id', how='left')

# Compute candidate_first_anomaly_timestamp as earliest non-null of the two
def pick_earliest(row):
    mts = row.get('earliest_metric_anomaly_timestamp')
    tts = row.get('earliest_trace_anomaly_timestamp')
    vals = []
    if not pd.isna(mts):
        vals.append(int(mts))
    if not pd.isna(tts):
        vals.append(int(tts))
    if not vals:
        return np.nan
    return int(min(vals))

c_df['candidate_first_anomaly_timestamp'] = c_df.apply(pick_earliest, axis=1)

# Reorder and round numeric breach ratios, keep compact columns
out_cols = [
    'cmdb_id',
    'earliest_metric_anomaly_timestamp','metric_kpi_name','metric_kpi_value','metric_breach_type','metric_breach_ratio',
    'earliest_trace_anomaly_timestamp','trace_name','trace_value','trace_breach_type','trace_breach_ratio',
    'candidate_first_anomaly_timestamp'
]

# Round numeric values for compactness
if 'metric_kpi_value' in c_df.columns:
    c_df['metric_kpi_value'] = c_df['metric_kpi_value'].round(6)
if 'metric_breach_ratio' in c_df.columns:
    c_df['metric_breach_ratio'] = c_df['metric_breach_ratio'].round(6)
if 'trace_value' in c_df.columns:
    c_df['trace_value'] = c_df['trace_value'].round(6)
if 'trace_breach_ratio' in c_df.columns:
    c_df['trace_breach_ratio'] = c_df['trace_breach_ratio'].round(6)

# Ensure timestamp columns are integers where present
for col in ['earliest_metric_anomaly_timestamp','earliest_trace_anomaly_timestamp','candidate_first_anomaly_timestamp']:
    if col in c_df.columns:
        c_df[col] = c_df[col].apply(lambda x: int(x) if (pd.notna(x) and float(x).is_integer()) else (int(x) if pd.notna(x) else np.nan))

# Final compact table (one row per candidate, max 7 rows)
final_candidate_summary = c_df.loc[:, out_cols]

final_candidate_summary

```
Out[6]:
```
```python
# Compose a plain-English compact summary for the candidate components using the previously computed table
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use final_candidate_summary from kernel
df = final_candidate_summary.copy()

def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

def fmt_val(v):
    if pd.isna(v):
        return "N/A"
    return str(v)

lines = []
for _, r in df.iterrows():
    cmdb = r['cmdb_id']
    m_ts = fmt_ts(r['earliest_metric_anomaly_timestamp'])
    m_name = r['metric_kpi_name'] if pd.notna(r['metric_kpi_name']) else "N/A"
    m_val = fmt_val(r['metric_kpi_value'])
    m_type = r['metric_breach_type'] if pd.notna(r['metric_breach_type']) else "N/A"
    m_ratio = fmt_val(r['metric_breach_ratio'])
    t_ts = fmt_ts(r['earliest_trace_anomaly_timestamp'])
    t_name = r['trace_name'] if pd.notna(r['trace_name']) else "N/A"
    t_val = fmt_val(r['trace_value'])
    t_type = r['trace_breach_type'] if pd.notna(r['trace_breach_type']) else "N/A"
    t_ratio = fmt_val(r['trace_breach_ratio'])
    first_ts = fmt_ts(r['candidate_first_anomaly_timestamp'])
    lines.append(
        f"{cmdb}: metric earliest={m_ts} -> [{m_name}] value={m_val}, type={m_type}, ratio={m_ratio}; "
        f"trace earliest={t_ts} -> [{t_name}] value={t_val}, type={t_type}, ratio={t_ratio}; "
        f"first_anomaly={first_ts}"
    )

summary = "Candidate anomaly summary (window 1647849600–1647851400 UTC+8):\n" + "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  earliest_metric_anomaly_timestamp                                    metric_kpi_name  metric_kpi_value metric_breach_type  metric_breach_ratio  earliest_trace_anomaly_timestamp                                 trace_name  trace_value trace_breach_type  trace_breach_ratio  candidate_first_anomaly_timestamp
0   paymentservice-0                         1647849600  container.node-6.container_network_transmit_pa...         136.00000                low             1.095588                      1.647850e+09     trace.from_checkoutservice-0.row_count     2.000000              high            1.000000                         1647849600
1   paymentservice-1                         1647849600        container.node-6.container_file_descriptors          22.00000               high             1.000000                      1.647850e+09     trace.from_checkoutservice-0.row_count     1.000000               low            1.000000                         1647849600
2   paymentservice-2                         1647849600        container.node-6.container_file_descriptors          22.00000               high             1.000000                      1.647850e+09  trace.from_checkoutservice-0.duration_p95     0.000187              high            1.012179                         1647849600
3  paymentservice2-0                         1647849600        container.node-6.container_file_descriptors          21.00000               high             1.000000                      1.647850e+09   trace.from_checkoutservice2-0.error_rate     0.000000              high                 NaN                         1647849600
4             node-6                         1647849600                                    system.mem.free        3084.00000                low             1.273346                               NaN                                        NaN          NaN               NaN                 NaN                         1647849600
5         frontend-1                         1647849600  container.node-5.container_network_receive_pac...        5866.00000               high             1.090943                      1.647850e+09          trace.to_adservice-1.duration_p95     0.000031              high            1.125227                         1647849600
6     cartservice2-0                         1647849600  container.node-5.container_network_receive_MB....           0.65436               high             3.043491                      1.647850e+09                       trace.self.row_count    73.000000               low            1.113014                         1647849600```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# timezone (UTC+8) per rules
tz = pytz.timezone('Asia/Shanghai')

# Candidates and window
target_cmdb = 'paymentservice-2'
start_ts = 1647849600
end_ts = 1647851400

# 1) Compute global P95/P05 and total_points per (cmdb_id, kpi_name) using the full metric series (global thresholds)
metric_group_stats = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        total_points_global='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter metric rows to the window and to the target cmdb_id
metric_window_target = metric_df[
    (metric_df['cmdb_id'] == target_cmdb) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# 3) Merge global thresholds into the window rows for the target
metric_win_with_thresh = metric_window_target.merge(
    metric_group_stats[metric_group_stats['cmdb_id'] == target_cmdb],
    on=['cmdb_id', 'kpi_name'],
    how='left',
    validate='m:1'
)

# 4) Select only anomalous rows where value >= P95 OR value <= P05
# (Use global P95/P05 computed above)
metric_win_with_thresh['is_anom'] = (
    (metric_win_with_thresh['value'] >= metric_win_with_thresh['P95']) |
    (metric_win_with_thresh['value'] <= metric_win_with_thresh['P05'])
)

anomalous = metric_win_with_thresh[metric_win_with_thresh['is_anom']].copy()

# 5) Aggregate anomalous rows by kpi_name with requested metrics
if anomalous.empty:
    # Create an empty DataFrame with the requested columns for compact display
    paymentservice2_summary = pd.DataFrame(columns=[
        'kpi_name','total_points_global','P95','P05','points_in_window',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value','min_value',
        'breach_ratio_high','breach_ratio_low'
    ])
else:
    agg = anomalous.groupby('kpi_name', dropna=False).agg(
        points_in_window=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max'),
        max_value=('value', 'max'),
        min_value=('value', 'min')
    ).reset_index()

    # Bring in global totals and thresholds
    global_info = metric_group_stats[metric_group_stats['cmdb_id'] == target_cmdb][[
        'kpi_name','total_points_global','P95','P05'
    ]]

    paymentservice2_summary = agg.merge(global_info, on='kpi_name', how='left')

    # Compute breach ratios per spec
    # breach_ratio_high = max_value / P95 (null if P95==0)
    paymentservice2_summary['breach_ratio_high'] = np.where(
        (paymentservice2_summary['P95'] != 0) & (~paymentservice2_summary['P95'].isna()),
        paymentservice2_summary['max_value'] / paymentservice2_summary['P95'],
        np.nan
    )

    # breach_ratio_low = P05 / min_value (null if min_value==0)
    paymentservice2_summary['breach_ratio_low'] = np.where(
        (paymentservice2_summary['min_value'] != 0) & (~paymentservice2_summary['min_value'].isna()),
        paymentservice2_summary['P05'] / paymentservice2_summary['min_value'],
        np.nan
    )

    # Reorder columns as requested
    paymentservice2_summary = paymentservice2_summary[[
        'kpi_name','total_points_global','P95','P05','points_in_window',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value','min_value',
        'breach_ratio_high','breach_ratio_low'
    ]]

    # Round numeric columns for compactness
    for col in ['P95','P05','max_value','min_value','breach_ratio_high','breach_ratio_low']:
        if col in paymentservice2_summary.columns:
            paymentservice2_summary[col] = paymentservice2_summary[col].round(6)

    # 6) Order rows by breach_ratio_high desc and limit to top 20
    paymentservice2_summary = paymentservice2_summary.sort_values(
        by=['breach_ratio_high','breach_ratio_low'], ascending=[False, False]
    ).head(20).reset_index(drop=True)

# Display compact result (one DataFrame variable)
paymentservice2_summary

```
Out[7]:
```
```python
# Plain-English compact summary for paymentservice-2 metric anomalies (based on the aggregated table)
summary = (
    "paymentservice-2 metric anomaly summary (window 1647849600–1647851400 UTC+8):\n\n"
    "Key findings:\n"
    "1) Extreme CPU breaches: container_cpu_usage_seconds and container_cpu_user_seconds show the largest breach ratios\n"
    "   • container_cpu_usage_seconds: breach_ratio_high ≈ 3002.48 (max ~24.79 vs P95 ≈ 0.008257), points_in_window=3\n"
    "   • container_cpu_user_seconds: breach_ratio_high ≈ 2814.5 (max ~28.145 vs P95 ≈ 0.01), points_in_window=11\n"
    "   These are orders-of-magnitude above P95 and should be investigated first (could indicate measurement/scaling issue or severe CPU spike).\n\n"
    "2) Other strong CPU/system signals: cpu cfs periods and system seconds are also highly elevated\n"
    "   • container_cpu_cfs_periods ratio ≈ 47.69, points_in_window=5\n"
    "   • container_cpu_system_seconds ratio ≈ 42.50, points_in_window=28\n\n"
    "3) Memory-related breaches (moderate to high):\n"
    "   • container_memory_cache breach_ratio ≈ 16.98 (31 anomalous samples)\n"
    "   • memory RSS / memory usage / working_set show moderate breach ratios (~1.49–1.88) across many samples\n"
    "   These indicate increased memory footprint and cache activity during the window.\n\n"
    "4) File descriptors, threads and some network/disk KPIs:\n"
    "   • file_descriptors ratio ≈ 1.45, threads ratio ≈ 2.53 (both with ~31 anomalous samples)\n"
    "   • network and fs usage show smaller breaches (~1.01–1.09), present but less extreme.\n\n"
    "5) Low-breach ratios: many P05 values are zero (or min values are zero), so severity_low is often unavailable (NaN).\n\n"
    "Timing and scope:\n"
    " - Most anomalous KPIs have anomalies spanning the window (earliest anomalies at or near 1647849600 and latest at 1647851400).\n"
    " - Several metrics have many anomalous samples in-window (e.g., memory_cache, threads, rss ~31 samples), indicating persistent abnormality.\n\n"
    "Actionable recommendation:\n"
    " - Prioritize investigating CPU metrics and the metric collection/scaling (extreme ratios suggest either real severe CPU saturation or a metric normalization/error).\n"
    " - Next, inspect memory growth (cache/RSS/working_set) and process-level activity (threads, file descriptor counts).\n"
    " - Check container CPU limits, cgroup throttling, and host/node resource pressure for node-6 since many container metrics reference node-6.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  total_points_global           P95           P05  points_in_window  earliest_anomaly_timestamp  latest_anomaly_timestamp     max_value     min_value  breach_ratio_high  breach_ratio_low
0        container.node-6.container_cpu_usage_seconds                   61  8.257000e-03  2.817000e-03                 3                  1647851100                1647851400  2.479296e+01  8.257000e-03        3002.482582          0.341190
1         container.node-6.container_cpu_user_seconds                   61  1.000000e-02  0.000000e+00                11                  1647849780                1647851400  2.814500e+01  0.000000e+00        2814.500000               NaN
2   container.node-6.container_memory_failures.con...                   61  9.900000e+01  0.000000e+00                 4                  1647850200                1647851400  9.157500e+03  0.000000e+00          92.500000               NaN
3   container.node-6.container_memory_failures.hie...                   61  9.900000e+01  0.000000e+00                 4                  1647850200                1647851400  9.157500e+03  0.000000e+00          92.500000               NaN
4          container.node-6.container_cpu_cfs_periods                   61  1.300000e+01  5.000000e+00                 5                  1647849840                1647851400  6.200000e+02  5.000000e+00          47.692308          1.000000
5       container.node-6.container_cpu_system_seconds                   61  1.000000e-02  0.000000e+00                28                  1647849600                1647851400  4.250000e-01  0.000000e+00          42.500000               NaN
6             container.node-6.container_memory_cache                   61  1.429504e+06  1.429504e+06                31                  1647849600                1647851400  2.427290e+07  1.429504e+06          16.979943          1.000000
7   mesh.destination.checkoutservice.paymentservic...                   61  6.300000e-03  2.625000e-03                 1                  1647851400                1647851400  6.240000e-02  6.240000e-02           9.904762          0.042067
8                  container.node-6.container_threads                   61  1.900000e+01  1.900000e+01                31                  1647849600                1647851400  4.800000e+01  1.900000e+01           2.526316          1.000000
9          container.node-6.container_memory_usage_MB                   61  4.516406e+01  4.514062e+01                29                  1647849600                1647851400  8.491211e+01  4.512695e+01           1.880081          1.000303
10              container.node-6.container_memory_rss                   61  3.399680e+07  3.399680e+07                31                  1647849600                1647851400  5.147443e+07  3.399680e+07           1.514096          1.000000
11   container.node-6.container_memory_working_set_MB                   61  4.285156e+01  4.282812e+01                29                  1647849600                1647851400  6.403711e+01  4.281445e+01           1.494394          1.000319
12        container.node-6.container_file_descriptors                   61  2.200000e+01  2.200000e+01                31                  1647849600                1647851400  3.200000e+01  2.200000e+01           1.454545          1.000000
13      container.node-6.container_memory_mapped_file                   61  1.486848e+06  1.486848e+06                31                  1647849600                1647851400  1.622016e+06  1.486848e+06           1.090909          1.000000
14  container.node-6.container_network_transmit_pa...                   61  2.395000e+02  1.605000e+02                 4                  1647849720                1647851220  2.600000e+02  1.525000e+02           1.085595          1.052459
15  container.node-6.container_network_receive_pac...                   61  3.370000e+02  2.415000e+02                 2                  1647849960                1647851220  3.640000e+02  2.335000e+02           1.080119          1.034261
16   container.node-6.container_fs_usage_MB./dev/vda1                   61  4.721484e+01  4.703516e+01                 4                  1647851220                1647851400  4.998438e+01  4.721484e+01           1.058658          0.996194
17  container.node-6.container_network_transmit_MB...                   61  3.325770e-01  3.114050e-01                 3                  1647849960                1647851220  3.380930e-01  3.112090e-01           1.016584          1.000631
18  container.node-6.container_network_receive_MB....                   61  3.123360e-01  2.134400e-02                 4                  1647849720                1647851220  3.150060e-01  2.034200e-02           1.008548          1.049249
19   container.node-6.container_fs_limit_MB./dev/vda1                   61  6.046307e+05  6.046307e+05                31                  1647849600                1647851400  6.046307e+05  6.046307e+05           1.000000          1.000000```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Parameters
target_cmdb = 'paymentservice-2'
start_ts = 1647849600
end_ts = 1647851400

# 1) Global P95/P05 and total_points per (cmdb_id, trace_name) using full series (must compute BEFORE filtering)
trace_group_stats = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        total_points='count',
        P95=lambda x: x.quantile(0.95),
        P05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Keep only groups for the target cmdb_id for downstream joins
trace_stats_target = trace_group_stats[trace_group_stats['cmdb_id'] == target_cmdb].copy()

# 2) Filter trace.csv to the given window and target cmdb_id
trace_window = trace_df[
    (trace_df['cmdb_id'] == target_cmdb) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# 3) Merge global thresholds into window rows and mark anomalies
trace_window = trace_window.merge(
    trace_stats_target[['cmdb_id','trace_name','total_points','P95','P05']],
    on=['cmdb_id','trace_name'],
    how='left',
    validate='m:1'
)

# Mark anomalies in-window using global thresholds
trace_window['is_high'] = trace_window['value'] >= trace_window['P95']
trace_window['is_low'] = trace_window['value'] <= trace_window['P05']
trace_window['is_anom'] = trace_window['is_high'] | trace_window['is_low']

# points_in_window per trace_name (count of rows in window, including non-anomalous)
points_in_window = (
    trace_window
    .groupby('trace_name', sort=False)['value']
    .agg(points_in_window='count')
    .reset_index()
)

# anomaly aggregates among anomaly rows in window (min timestamp, count, max/min among anomaly rows)
anom_rows = trace_window[trace_window['is_anom']].copy()

if not anom_rows.empty:
    anom_agg = (
        anom_rows
        .groupby('trace_name', sort=False)
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_value=('value','max'),
            min_value=('value','min')
        )
        .reset_index()
    )
else:
    anom_agg = pd.DataFrame(columns=['trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','min_value'])

# Determine existence of high/low anomalies per trace_name (needed for severity rules)
high_exist = (
    trace_window[trace_window['is_high']]
    .groupby('trace_name', sort=False)['value']
    .agg(has_high='count')
    .reset_index()
) if not trace_window[trace_window['is_high']].empty else pd.DataFrame(columns=['trace_name','has_high'])

low_exist = (
    trace_window[trace_window['is_low']]
    .groupby('trace_name', sort=False)['value']
    .agg(has_low='count')
    .reset_index()
) if not trace_window[trace_window['is_low']].empty else pd.DataFrame(columns=['trace_name','has_low'])

# 4) Merge aggregates with global stats for target cmdb_id
merged = (
    trace_stats_target
    .rename(columns={'total_points':'total_points_global'})
    .merge(points_in_window, on='trace_name', how='left')
    .merge(anom_agg, on='trace_name', how='left')
    .merge(high_exist, on='trace_name', how='left')
    .merge(low_exist, on='trace_name', how='left')
)

# Fill missing numeric counts with zeros where appropriate
merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)
merged['has_high'] = merged.get('has_high').fillna(0).astype(int) if 'has_high' in merged.columns else 0
merged['has_low'] = merged.get('has_low').fillna(0).astype(int) if 'has_low' in merged.columns else 0

# Ensure earliest/max/min are NaN when no anomalies
no_anom_mask = merged['anomaly_count'] == 0
for c in ['earliest_anomaly_timestamp','max_value','min_value']:
    if c in merged.columns:
        merged.loc[no_anom_mask, c] = np.nan

# 4) Compute severity_high and severity_low per spec
# severity_high = max_value / P95 (null if P95==0 or no high anomalies)
merged['severity_high'] = np.where(
    (merged['has_high'] > 0) & (merged['P95'] != 0) & (~merged['P95'].isna()) & (~merged['max_value'].isna()),
    merged['max_value'] / merged['P95'],
    np.nan
)

# severity_low = P05 / min_value (null if min_value==0 or no low anomalies)
merged['severity_low'] = np.where(
    (merged['has_low'] > 0) & (merged['min_value'].notna()) & (merged['min_value'] != 0),
    merged['P05'] / merged['min_value'],
    np.nan
)

# 5) Prepare final output columns and ordering
output_cols = [
    'trace_name',
    'total_points', # per spec they asked total_points (global) — our trace_stats_target used 'total_points'
    'P95','P05',
    'points_in_window','anomaly_count','earliest_anomaly_timestamp',
    'max_value','min_value','severity_high','severity_low'
]

# Ensure the 'total_points' column exists (rename if necessary)
if 'total_points' in merged.columns:
    merged = merged.rename(columns={'total_points':'total_points'})  # already present
else:
    merged['total_points'] = merged['total_points_global']

# Select and sort
result_trace_ps2 = (
    merged.loc[:, ['trace_name','total_points','P95','P05','points_in_window','anomaly_count',
                   'earliest_anomaly_timestamp','max_value','min_value','severity_high','severity_low']]
    .sort_values(by=['anomaly_count','severity_high'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Round numeric columns for compactness
for col in ['P95','P05','max_value','min_value','severity_high','severity_low']:
    if col in result_trace_ps2.columns:
        result_trace_ps2[col] = result_trace_ps2[col].round(6)

# Final compact table
result_trace_ps2

```
Out[8]:
```
```python
# Plain-English summary of the paymentservice-2 trace analysis (compact)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use result_trace_ps2 computed earlier in the kernel
df = result_trace_ps2.copy()

# Helper to format timestamps into UTC+8 human time
def fmt_ts(ts):
    try:
        if pd.isna(ts):
            return "N/A"
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except:
        return str(ts)

# High-level observations
total_groups = len(df)
# Many groups have anomaly_count == points_in_window meaning persistent anomalies across the window
persistent_groups = df[df['anomaly_count'] == df['points_in_window']].shape[0]

# Top trace groups by severity_high (non-null)
top_sev = df[df['severity_high'].notna()].sort_values('severity_high', ascending=False).head(6)

top_lines = []
for _, r in top_sev.iterrows():
    top_lines.append(
        f"{r['trace_name']} | P95={r['P95']} | max={r['max_value']} | severity_high={r['severity_high']} | "
        f"anomaly_count={int(r['anomaly_count'])} | earliest={fmt_ts(r['earliest_anomaly_timestamp'])}"
    )

summary = (
    "paymentservice-2 trace summary (window 1647849600–1647851400, UTC+8):\n\n"
    f"- Examined {total_groups} trace groups for paymentservice-2. {persistent_groups} groups were anomalous on every sampled point in-window (points_in_window == anomaly_count).\n\n"
    "- Key patterns:\n"
    "  • Row-count increases: Several row_count traces (e.g., trace.from_checkoutservice-*.row_count) show max=3 vs P95=2 (severity_high=1.5), indicating elevated request counts/traffic into paymentservice-2.\n"
    "  • Latency spikes: Duration metrics show notable increases. The highest severity is:\n"
    f"    - {top_lines[0] if len(top_lines)>0 else 'None'}\n"
    "    Additional elevated duration traces (top by severity):\n"
    + ("\n    - " + "\n    - ".join(top_lines[1:]) if len(top_lines) > 1 else "\n    - (no additional high-severity duration traces)") + "\n\n"
    "  • Error-rate traces often have P95 == 0 and max == 0 so severity_low/high are not computable (NaN) — these don't show useful numeric breach ratios here.\n\n"
    "- Interpretation & recommendation:\n"
    "  1) The systemic pattern is persistent anomalies across the window for many traces, not single spikes.\n  2) Prioritize investigating the checkout → payment request path (trace.from_checkoutservice.* and trace.to_paymentservice.*): increased row counts and elevated durations suggest higher load and increased latency on paymentservice-2.\n  3) Focus immediate debugging on the traces with largest severity_high (listed above): they show the largest multiple over normal P95 latency.\n  4) Correlate these trace findings with the metric anomalies already found for paymentservice-2 (CPU/memory signals) to identify whether the issue is resource saturation, downstream slowdown, or instrumentation/normalization problems.\n\n"
    "Timestamps above are in UTC+8."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  total_points       P95       P05  points_in_window  anomaly_count  earliest_anomaly_timestamp  max_value  min_value  severity_high  severity_low
0       trace.from_checkoutservice-2.row_count            30  2.000000  1.000000                30             30                1.647850e+09   3.000000   1.000000       1.500000      1.000000
1       trace.from_checkoutservice-0.row_count            30  2.000000  1.000000                30             30                1.647850e+09   2.000000   1.000000       1.000000      1.000000
2      trace.from_checkoutservice-0.error_rate            30  0.000000  0.000000                30             30                1.647850e+09   0.000000   0.000000            NaN           NaN
3      trace.from_checkoutservice-2.error_rate            30  0.000000  0.000000                30             30                1.647850e+09   0.000000   0.000000            NaN           NaN
4       trace.from_checkoutservice-1.row_count            27  2.000000  1.000000                27             27                1.647850e+09   3.000000   1.000000       1.500000      1.000000
5      trace.from_checkoutservice-1.error_rate            27  0.000000  0.000000                27             27                1.647850e+09   0.000000   0.000000            NaN           NaN
6   trace.from_checkoutservice-2.duration_mean            30  0.000169  0.000142                30              5                1.647850e+09   0.000189   0.000141       1.117682      1.007092
7    trace.from_checkoutservice-1.duration_p95            27  0.000199  0.000142                27              4                1.647851e+09   0.000697   0.000140       3.504274      1.016429
8   trace.from_checkoutservice-1.duration_mean            27  0.000188  0.000142                27              4                1.647850e+09   0.000361   0.000140       1.923282      1.016429
9    trace.from_checkoutservice-2.duration_p95            30  0.000169  0.000142                30              4                1.647850e+09   0.000209   0.000141       1.235955      1.010284
10  trace.from_checkoutservice-0.duration_mean            30  0.000175  0.000143                30              4                1.647850e+09   0.000192   0.000142       1.099342      1.006338
11   trace.from_checkoutservice-0.duration_p95            30  0.000185  0.000143                30              4                1.647850e+09   0.000192   0.000142       1.039242      1.006338```
```